[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apartsin/DLCourseHIT/blob/main/notebooks/week06.ipynb)

# Week 6: Optimization
**Introduction to Deep Learning (HIT)** &middot; Part II: Training Infrastructure

Gradient descent; SGD, momentum, and Adam; learning rates and optimization dynamics.

**Instructor practice notebook.** Run these live demonstrations during the 2-hour practice lesson. The student homework is the weekly lab.

## Goals for the practice lesson

- Understand SGD, momentum, and Adam.
- Reason about learning rates and convergence.
- Tune an optimizer to a target.

## Setup
Run this first. On Colab, set Runtime > Change runtime type > GPU for the later weeks.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
print('PyTorch', torch.__version__, '| device:', device)

## Demonstration 1
Train the same model with SGD, momentum, and Adam, and compare the curves.

In [ ]:
# Same model and data, three optimizers
torch.manual_seed(0)
X = torch.randn(200, 5); y = X @ torch.randn(5, 1) + 0.1 * torch.randn(200, 1)
def run(name):
    torch.manual_seed(0); m = nn.Linear(5, 1); f = nn.MSELoss()
    o = {"SGD": torch.optim.SGD(m.parameters(), lr=0.05),
         "Momentum": torch.optim.SGD(m.parameters(), lr=0.05, momentum=0.9),
         "Adam": torch.optim.Adam(m.parameters(), lr=0.05)}[name]
    h = []
    for _ in range(60):
        o.zero_grad(); l = f(m(X), y); l.backward(); o.step(); h.append(l.item())
    return h
for name in ["SGD", "Momentum", "Adam"]:
    plt.plot(run(name), label=name)
plt.legend(); plt.xlabel("step"); plt.ylabel("loss"); plt.show()

## Demonstration 2
Sweep three learning rates live and read the resulting curves.

In [ ]:
# Learning-rate sweep
for lr in [0.001, 0.05, 0.5]:
    torch.manual_seed(0); m = nn.Linear(5, 1); o = torch.optim.SGD(m.parameters(), lr=lr); f = nn.MSELoss()
    h = []
    for _ in range(60):
        o.zero_grad(); l = f(m(X), y); l.backward(); o.step(); h.append(l.item())
    plt.plot(h, label=f"lr={lr}")
plt.yscale("log"); plt.legend(); plt.xlabel("step"); plt.ylabel("loss (log)"); plt.show()

## Demonstration 3
Add a learning-rate schedule and show its effect.

In [ ]:
# A learning-rate schedule (step decay)
m = nn.Linear(5, 1); o = torch.optim.SGD(m.parameters(), lr=0.1); f = nn.MSELoss()
sched = torch.optim.lr_scheduler.StepLR(o, step_size=20, gamma=0.3)
lrs = []
for _ in range(60):
    o.zero_grad(); f(m(X), y).backward(); o.step(); sched.step()
    lrs.append(o.param_groups[0]["lr"])
plt.plot(lrs); plt.xlabel("step"); plt.ylabel("learning rate"); plt.title("StepLR"); plt.show()

---
Student materials for this week: the lab handout (`labs/week06.html`) and the curated references (`references/week06.html`) in the course site.